# 01. Shared service credential

The agent calls the tool with a single static API key on every request,
for every user. The tool has no idea who is asking.

This is where most early agentic prototypes start. The agent is its own
trusted system, and "auth" between agent and tool is a shared secret that
proves "I am the agent". The tool returns whatever the agent asks for, and
the agent is responsible for showing the user the right subset of what came
back.

Notebooks 2 through 8 each fix one thing that's broken about this baseline.

## What this pattern looks like on the wire

```
   alice -+
          |
   bob   -+--->  agent  -- X-API-Key: dev-shared-api-key -->  expense-service
          |
   dave  -+
```

The arrows from the users to the agent might be HTTP, a chat UI, a Slack
bot. That part isn't the point. The point is the right-hand arrow: exactly
the same header on every request, regardless of which user prompted the agent.

In [1]:
from shared.agent import Agent
from shared.auth import ServiceCredentialAuth
from shared.tools import ALL_TOOLS
from shared.display import run_as, show_what_tool_saw
from shared.config import EXPENSE_SERVICE_URL

strategy = ServiceCredentialAuth()
agent = Agent(strategy=strategy, tools=ALL_TOOLS)
print(f"strategy: {strategy.name}")

strategy: service_credential


## Three users, same prompt

Watch the answers carefully. The agent has no information about who is
asking. Whatever filtering happens is done by the LLM reading the data, not
by the system.

In [2]:
result_alice = run_as("alice", "What expenses do I have?", agent)

╭──────────────────────────────────╮
│ [alice] What expenses do I have? │
╰──────────────────────────────────╯

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool         ┃ args                 ┃ status ┃ result                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses │ {'department': None} │    200 │ {"_status": 200, "identity_method": "api_key",             │
│     │              │                      │        │ "identity_detail": "shared service credential, no user     │
│     │              │                      │        │ identity", "count": 8, "expenses": [{"id": 1, "user_id":   │
│     │              │                      │        │ "alice", "department": "engineering", "amount": 42.5,      │
│     │              │                      │        │ "category": "software", "description": "JetBrains AI       │
│     │              │                      │        │ assistant subscription", "status":                         │
└─────┴──────────────┴──────────────────────┴────────┴────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are the expenses visible under your current access:                                                        │
│                                                                                                                 │
│ - 1) id 1 | alice | engineering | $42.50 | software | JetBrains AI assistant subscription | approved            │
│ - 2) id 2 | alice | engineering | $156.00 | travel | Train ticket to client offsite | approved                  │
│ - 3) id 3 | alice | engineering | $89.00 | books | Designing Data-Intensive Applications | approved             │
│ - 4) id 4 | alice | engineering | $1,450.00 | hardware | External 4K monitor | pending                          │
│ - 5) id 5 | bob | engineering | $320.00 | training | OAuth 2.0 deep-dive workshop | approved                    │
│ - 6) id 6 | bob | engineering | $67.00 | meals | Team lunch after the migration shipped | approved              │
│ - 7) id 7 | dave | platform | $980.00 | training | KubeCon ticket | approved                                    │
│ - 8) id 8 | dave | platform | $240.00 | software | Datadog license seat | approved                              │
│                                                                                                                 │
│ Notes:                                                                                                          │
│ - One item is pending (id 4).                                                                                   │
│ - If you want to see only your own expenses, please log in with your account (or let me know your user identity │
│ and I’ll fetch accordingly). I can also filter by department if you’d like.                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [3]:
result_bob = run_as("bob", "What expenses do my team have?", agent)

╭──────────────────────────────────────╮
│ [bob] What expenses do my team have? │
╰──────────────────────────────────────╯

╭─────────────────────────────────────── answer ────────────────────────────────────────╮
│ I can pull your team’s expenses, but I need the department name.                      │
│                                                                                       │
│ - If you want only your own expenses, I can fetch those now (no department filter).   │
│ - For your team, please provide the department name (e.g., “Engineering” or “Sales”). │
╰───────────────────────────────────────────────────────────────────────────────────────╯

In [4]:
result_dave = run_as("dave", "Show me all the expenses across the company.", agent)

╭─────────────────────────────────────────────────────╮
│ [dave] Show me all the expenses across the company. │
╰─────────────────────────────────────────────────────╯

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool         ┃ args                 ┃ status ┃ result                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses │ {'department': None} │    200 │ {"_status": 200, "identity_method": "api_key",             │
│     │              │                      │        │ "identity_detail": "shared service credential, no user     │
│     │              │                      │        │ identity", "count": 8, "expenses": [{"id": 1, "user_id":   │
│     │              │                      │        │ "alice", "department": "engineering", "amount": 42.5,      │
│     │              │                      │        │ "category": "software", "description": "JetBrains AI       │
│     │              │                      │        │ assistant subscription", "status":                         │
└─────┴──────────────┴──────────────────────┴────────┴────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are all 8 expenses across the company:                                                                     │
│                                                                                                                 │
│ - ID 1 | User: alice | Dept: engineering | Category: software | Amount: $42.50 | Description: JetBrains AI      │
│ assistant subscription | Status: approved                                                                       │
│ - ID 2 | User: alice | Dept: engineering | Category: travel | Amount: $156.00 | Description: Train ticket to    │
│ client offsite | Status: approved                                                                               │
│ - ID 3 | User: alice | Dept: engineering | Category: books | Amount: $89.00 | Description: Designing            │
│ Data-Intensive Applications | Status: approved                                                                  │
│ - ID 4 | User: alice | Dept: engineering | Category: hardware | Amount: $1,450.00 | Description: External 4K    │
│ monitor | Status: pending                                                                                       │
│ - ID 5 | User: bob | Dept: engineering | Category: training | Amount: $320.00 | Description: OAuth 2.0          │
│ deep-dive workshop | Status: approved                                                                           │
│ - ID 6 | User: bob | Dept: engineering | Category: meals | Amount: $67.00 | Description: Team lunch after the   │
│ migration shipped | Status: approved                                                                            │
│ - ID 7 | User: dave | Dept: platform | Category: training | Amount: $980.00 | Description: KubeCon ticket |     │
│ Status: approved                                                                                                │
│ - ID 8 | User: dave | Dept: platform | Category: software | Amount: $240.00 | Description: Datadog license seat │
│ | Status: approved                                                                                              │
│                                                                                                                 │
│ Total: $3,344.50 across 8 expenses.                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## What did the tool actually see?

The agent worked hard to format the right answer for each user. Here is
what the service actually got on its most recent request.

In [5]:
show_what_tool_saw(EXPENSE_SERVICE_URL)

╭──────── expense-service /debug/last-request ─────────╮
│ method:  api_key                                     │
│ user_id: <none>                                      │
│ detail:  shared service credential, no user identity │
│                                                      │
╰──────────────────────────────────────────────────────╯

{'method': 'api_key', 'detail': 'shared service credential, no user identity'}

## Tradeoffs

- **Authz lives nowhere.** Or rather, in the LLM's prompt-following ability,
  which is not a security boundary.
- **The tool sees no real user.** Method on the service side is `api_key`.
  The service has no idea whether alice or bob or dave asked.
- **No cryptographic proof of identity.** Anyone with the API key is the agent.
- **No least privilege.** The API key is fully privileged.
- **Audit trail:** "agent did X" is the most you can say. You cannot answer
  "did alice access bob's expenses?" from the service logs alone.
- **Prompt injection blast radius:** an attacker who gets the agent to do
  the wrong thing has full read access to every user's data, because the
  tool has no scoping.

What does work: the agent is simple, and the service is simple. Nothing
depends on a complex identity infrastructure. For a single-tenant internal
tool with no privacy or compliance constraints, this might genuinely be enough.

## What's still missing

The agent received three different prompts from three different users, and
in each case the LLM tried its best to filter the result *in its answer*.
The service returned the same data regardless of who asked. We're trusting
the LLM not to leak information across users, and the LLM is not a security
boundary.

The most direct fix is to push some awareness of "who is asking" from the
user, through the agent, into the tool call itself. Different patterns do
this differently, and each has its own failure modes.

Notebook 02 takes the smallest possible step: the agent reads the user's
identity claims and uses them to construct narrower tool calls. The tool
still has no idea who's asking, but at least the call is scoped before it
goes out.